# 04 — German CLIP Semantic Closeness (within categories)

Same logic as `01_english_clip_distance.ipynb`, run on the German-translated selection —
groups by category and selects the N semantically **closest** words within each category,
using the same multilingual CLIP model so English and German embeddings are directly
comparable.


In [1]:
import os
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.metrics import pairwise_distances
from sklearn.manifold import MDS


/home/hivrim8h/projects/EventSegmentation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===== Config =====
CSV_PATH = "output/german_selection.csv"
WORD_COL = "word_de"
GROUP_COL = "category"
DEVICE = "cpu"
SAVE_DIR = "output/mds_plots_de_category"
ZIPF_COL = "ZipfSUBTLEX"
N_SELECT = 3


In [3]:
df = pd.read_csv(CSV_PATH)
df[WORD_COL] = df[WORD_COL].astype(str).str.strip()
groups = {g: grp[WORD_COL].tolist() for g, grp in df.groupby(GROUP_COL)}
print({g: len(v) for g, v in sorted(groups.items(), key=lambda x: -len(x[1]))})


{'food': 73, 'clothing': 35, 'container': 27, 'arts and crafts supply': 21, 'electronic device': 18, 'sports equipment': 17, 'musical instrument': 16, 'personal hygiene item': 16, 'home decor': 14, 'body part': 12, 'hardware': 12, 'kitchen tool': 11, 'clothing accessory': 7, 'office supply': 7, 'part of car': 7, 'medical equipment': 6, 'scientific equipment': 6, 'tool': 6, 'toy': 6, 'animal': 5, 'breakfast food': 5, 'construction equipment': 5, 'game': 5, 'condiment': 4, 'fastener': 4, 'jewelry': 3, 'plant': 3, 'safety equipment': 3, 'school supply': 3, 'candy': 2, 'dessert': 2, 'drink': 2, 'headwear': 2, 'lighting': 2, 'footwear': 1, 'furniture': 1, 'garden tool': 1, 'protective clothing': 1}


In [4]:
model = SentenceTransformer("sentence-transformers/clip-ViT-B-32-multilingual-v1", device=DEVICE)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3191.21it/s]


In [5]:
all_words = sorted(set(df[WORD_COL]))
emb_all = model.encode(all_words, normalize_embeddings=True)
word2vec = {w: emb_all[i] for i, w in enumerate(all_words)}


In [6]:
def find_most_close_set(words, D, n=3):
    best_sum = np.inf
    best_combo = None
    for combo in itertools.combinations(range(len(words)), n):
        first_letters = {words[i][0].lower() for i in combo}
        if len(first_letters) < n:
            continue
        total_dist = sum(D[i, j] for i, j in itertools.combinations(combo, 2))
        if total_dist < best_sum:
            best_sum = total_dist
            best_combo = combo
    if best_combo is None:
        for combo in itertools.combinations(range(len(words)), n):
            total_dist = sum(D[i, j] for i, j in itertools.combinations(combo, 2))
            if total_dist < best_sum:
                best_sum = total_dist
                best_combo = combo
    return [words[i] for i in best_combo], best_sum


In [7]:
def plot_group(words, emb, group_id, selected_words, mean_dist, save_dir=""):
    if len(words) < 2:
        print(f"Category '{group_id}' has too few words, skipping plot")
        return
    D = pairwise_distances(emb, metric="cosine")
    coords = MDS(n_components=2, dissimilarity="precomputed", random_state=42, n_init=4).fit_transform(D)
    sel_set = set(selected_words)

    plt.figure(figsize=(6, 5))
    idx_other = [i for i, w in enumerate(words) if w not in sel_set]
    plt.scatter(coords[idx_other, 0], coords[idx_other, 1], c="#9aa0a6", s=30, label="other words")
    idx_sel = [i for i, w in enumerate(words) if w in sel_set]
    plt.scatter(coords[idx_sel, 0], coords[idx_sel, 1], c="#1a73e8", s=60, label="closest words")
    for i in idx_sel:
        plt.annotate(words[i], (coords[i, 0], coords[i, 1]),
                     xytext=(4, 4), textcoords="offset points", fontsize=9, color="#1a73e8")

    plt.title(f"{group_id} (DE) — MDS\nAvg_Distance={mean_dist:.3f}")
    plt.xlabel("MDS-1"); plt.ylabel("MDS-2")
    plt.legend(frameon=False); plt.grid(alpha=0.2)
    plt.tight_layout()

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        safe_name = str(group_id).replace(" ", "_").replace("/", "-")
        plt.savefig(os.path.join(save_dir, f"{safe_name}_mds_de.png"), dpi=150)
        plt.close()
    else:
        plt.show()


In [8]:
results = []
skipped = []
for g in sorted(groups.keys()):
    words = groups[g]
    if len(words) < N_SELECT:
        skipped.append((g, len(words)))
        continue
    emb = np.stack([word2vec[w] for w in words])
    D = pairwise_distances(emb, metric="cosine")
    selected_words, total_dist = find_most_close_set(words, D, n=N_SELECT)
    mean_dist = total_dist / (N_SELECT * (N_SELECT - 1) / 2)
    plot_group(words, emb, g, selected_words, mean_dist, save_dir=SAVE_DIR)

    sel_df = df[df[WORD_COL].isin(selected_words) & (df[GROUP_COL] == g)][
        [WORD_COL, "Word", "GermanTranslation", ZIPF_COL, "category"]
    ]
    for _, r in sel_df.iterrows():
        results.append({
            "category": g,
            "word_de": r[WORD_COL],
            "Word_english": r["Word"],
            "GermanTranslation": r["GermanTranslation"],
            ZIPF_COL: r[ZIPF_COL],
            "category_total_distance": total_dist,
            "category_mean_distance": mean_dist
        })

res_df = pd.DataFrame(results)
print(res_df)
if skipped:
    print(f"\nSkipped {len(skipped)} categories with fewer than {N_SELECT} candidate words:")
    for g, n in skipped:
        print(f"  {g}: {n} words")


/home/hivrim8h/projects/EventSegmentation/.venv/lib/python3.12/site-packages/sklearn/manifold/_mds.py:735: FutureWarning: The default value of `init` will change from 'random' to 'classical_mds' in 1.10. To suppress this warning, provide some value of `init`.
  warnings.warn(
/home/hivrim8h/projects/EventSegmentation/.venv/lib/python3.12/site-packages/sklearn/manifold/_mds.py:752: FutureWarning: The `dissimilarity` parameter is deprecated and will be removed in 1.10. Use `metric` instead.
  warnings.warn(
/home/hivrim8h/projects/EventSegmentation/.venv/lib/python3.12/site-packages/sklearn/manifold/_mds.py:735: FutureWarning: The default value of `init` will change from 'random' to 'classical_mds' in 1.10. To suppress this warning, provide some value of `init`.
  warnings.warn(
/home/hivrim8h/projects/EventSegmentation/.venv/lib/python3.12/site-packages/sklearn/manifold/_mds.py:752: FutureWarning: The `dissimilarity` parameter is deprecated and will be removed in 1.10. Use `metric` inst

                  category      word_de Word_english GermanTranslation  \
0                   animal        katze          cat             Katze   
1                   animal         huhn      chicken              Huhn   
2                   animal     schnecke        snail          Schnecke   
3   arts and crafts supply    bleistift       pencil         Bleistift   
4   arts and crafts supply       kreide        chalk            Kreide   
..                     ...          ...          ...               ...   
82                    tool  essstäbchen   chopsticks       Essstäbchen   
83                    tool        rakel     squeegee             Rakel   
84                     toy        puppe         doll             Puppe   
85                     toy    spielzeug          toy         Spielzeug   
86                     toy      drachen         kite           Drachen   

    ZipfSUBTLEX  category_total_distance  category_mean_distance  
0      4.641288                 0.060707    

In [9]:
res_df.to_csv("output/closest_selection_german.csv", index=False, encoding="utf-8-sig")
print(f"Saved {len(res_df)} selected words across {res_df['category'].nunique()} categories "
      f"to output/closest_selection_german.csv")


Saved 87 selected words across 29 categories to output/closest_selection_german.csv
